# Movie Recommender PAD — Etapa M: BGE-Large (GPU)

Este notebook executa o pipeline completo com `BAAI/bge-large-en-v1.5` em todos os 62.423 filmes do MovieLens ml-25m.

**Requer GPU:** Ativar em *Editar > Configuracoes do notebook > Acelerador de hardware > T4 GPU*.

**Antes de rodar:** faca upload da pasta `ml-25m/` para o Google Drive em `Meu Drive/ml-25m/` (ou ajuste `DATA_DIR` na celula de configuracao abaixo).

In [1]:
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('GPU detectada:', result.stdout.strip())
else:
    print('AVISO: GPU nao detectada. Va em Editar > Configuracoes do notebook > T4 GPU.')

GPU detectada: Tesla T4, 15360 MiB


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!git clone https://github.com/villarantonio/Movie-Recommender-PAD.git
%cd Movie-Recommender-PAD

Cloning into 'Movie-Recommender-PAD'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 32 (delta 7), reused 16 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 23.31 KiB | 11.65 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/Movie-Recommender-PAD


In [4]:
!pip install -q sentence-transformers scikit-learn

In [5]:
import os

DATA_DIR   = '/content/drive/MyDrive/ml-25m'
CACHE_PATH = '/content/drive/MyDrive/embeddings_bge_large.npy'
MODEL_NAME = 'BAAI/bge-large-en-v1.5'

assert os.path.exists(DATA_DIR), (
    f'Pasta nao encontrada: {DATA_DIR}\n'
    'Faca upload da pasta ml-25m/ para o Google Drive e ajuste DATA_DIR acima.'
)
print('Dados encontrados em:', DATA_DIR)
print('Cache de embeddings :', CACHE_PATH)
print('Modelo              :', MODEL_NAME)

Dados encontrados em: /content/drive/MyDrive/ml-25m
Cache de embeddings : /content/drive/MyDrive/embeddings_bge_large.npy
Modelo              : BAAI/bge-large-en-v1.5


## 1. Preprocessamento

In [6]:
from src.preprocess import build_soup

df = build_soup(DATA_DIR)
print(f'Total de filmes: {len(df)}')
df.head(3)

[preprocess] 13816 movies with genome-tags, 48607 movies using genre-only soup.
Total de filmes: 62423


,movieId,title,genres,soup,has_genome_tags,imdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3d action adventure affectionate animal movie ...,True,114709
1,2,Jumanji (1995),Adventure|Children|Fantasy,action adaptation adventure animals bad cgi ba...,True,113497
2,3,Grumpier Old Men (1995),Comedy|Romance,comedy crappy sequel destiny good good sequel ...,True,113228


## 2. Embeddings — bge-large-en-v1.5 em todos os 62k filmes

Na T4 (16 GB VRAM) com `batch_size=256` leva cerca de **3 a 5 minutos**.
O cache e salvo no Drive; execucoes seguintes carregam instantaneamente.

In [7]:
import numpy as np
import time

if os.path.exists(CACHE_PATH):
    print(f'[embeddings] Carregando cache: {CACHE_PATH}')
    embeddings = np.load(CACHE_PATH)
    print(f'[embeddings] Shape: {embeddings.shape}')
else:
    from sentence_transformers import SentenceTransformer

    print(f'[embeddings] Carregando modelo {MODEL_NAME} ...')
    model = SentenceTransformer(MODEL_NAME, device='cuda')

    print(f'[embeddings] Gerando embeddings para {len(df)} filmes ...')
    t0 = time.time()
    embeddings = model.encode(
        df['soup'].tolist(),
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True,
        device='cuda',
    )
    elapsed = time.time() - t0
    print(f'[embeddings] Concluido em {elapsed:.1f}s. Shape: {embeddings.shape}')
    np.save(CACHE_PATH, embeddings)
    print(f'[embeddings] Cache salvo em: {CACHE_PATH}')

[embeddings] Carregando cache: /content/drive/MyDrive/embeddings_bge_large.npy
[embeddings] Shape: (62423, 1024)


## 3. Benchmark — The Prestige (2006)

In [8]:
from src.recommender import build_cosine_matrix
from src.benchmark import run_benchmark

cosine_sim = build_cosine_matrix(embeddings)
run_benchmark(df, cosine_sim, model_name=MODEL_NAME)

[recommender] 62423 movies detected — using on-demand similarity (full matrix would require ~16 GB RAM).

BENCHMARK: Prestige, The (2006)

[BAAI/bge-large-en-v1.5] Top-10 recommendations:
Rank  Title                                         Genres                              Score
----------------------------------------------------------------------------------------------------
1     The Hound of the Baskervilles (1988)          Crime|Drama|Horror|Mystery          0.8831
2     My Cousin Rachel (2017)                       Drama|Romance                       0.8700
3     From Hell (2001)                              Crime|Horror|Mystery|Thriller       0.8668
4     Hakuchi (1951)                                Crime|Drama|Romance                 0.8629
5     Oliver Twist (1948)                           Adventure|Crime|Drama               0.8604
6     Camille Claudel (1988)                        Drama                               0.8596
7     Magnificent Ambersons, The (1942)        

## 4. Explorar outras recomendacoes (opcional)

In [9]:
from src.recommender import get_recommendations

TITULO = 'Inception (2010)'

recs = get_recommendations(TITULO, df, cosine_sim, top_n=10)
print(f'Top-10 similares a "{TITULO}":')
print(recs.to_string(index=False))

Top-10 similares a "Inception (2010)":
 rank                                 title                                   genres  similarity_score
    1                    Matrix, The (1999)                   Action|Sci-Fi|Thriller            0.9068
    2                    Source Code (2011)     Action|Drama|Mystery|Sci-Fi|Thriller            0.9062
    3                   Interstellar (2014)                              Sci-Fi|IMAX            0.9034
    4                     Mr. Nobody (2009)             Drama|Fantasy|Romance|Sci-Fi            0.9032
    5      Run Lola Run (Lola rennt) (1998)                             Action|Crime            0.9022
    6        Nikitich and The Dragon (2006) Adventure|Animation|Comedy|Drama|Fantasy            0.9013
    7                Minority Report (2002)     Action|Crime|Mystery|Sci-Fi|Thriller            0.8980
    8             Mad Max: Fury Road (2015)         Action|Adventure|Sci-Fi|Thriller            0.8970
    9                     Ex Machi

In [10]:
titulo_busca = 'Prestige, The (2006)'
row = df[df['title'] == titulo_busca].iloc[0]
print(f'Titulo : {row["title"]}')
print(f'Generos: {row["genres"]}')
print(f'Soup   : {row["soup"][:300]}...')

Titulo : Prestige, The (2006)
Generos: Drama|Mystery|Sci-Fi|Thriller
Soup   : 19th century adaptation adapted from:book alternate endings anti-hero atmospheric bad ending based on a book based on book beautifully filmed betrayal better than expected catastrophe cerebral character study cinematography clever clones cloning complex complicated complicated plot con men confusing...
